In [1]:
import chromadb
from sentence_transformers import SentenceTransformer

In [2]:
import os
import re

from dotenv import find_dotenv, load_dotenv
from huggingface_hub import login
from sentence_transformers import SentenceTransformer

# توكن HuggingFace من .env فقط
load_dotenv(find_dotenv())
login(token=os.environ["huggingface_Access_Tokens"])

# توحيد النص قبل الترميز (نفس ما استخدم في بناء الإمبدنجز)
TASHKEEL = "".join(chr(c) for c in list(range(0x064B, 0x0660)) + [0x0670])


def normalize_ar(text):
    text = text.translate(str.maketrans("", "", TASHKEEL))
    return re.sub(r"\s+", " ", text).strip()


MODEL_NAME = "omarelshehy/Arabic-Retrieval-v1.0"

embedding_model = SentenceTransformer(MODEL_NAME, device="cpu")

print("Embedding model loaded successfully!")
print("Embedding dimension:", embedding_model.get_embedding_dimension())


Embedding model loaded successfully!
Embedding dimension: 768


In [3]:
chroma_client = chromadb.PersistentClient(
    path="../chroma_db"
)

collection = chroma_client.get_collection(
    name="arbaeen_nawawi_small"
)

print("Collection:", collection.name)
print("Documents:", collection.count())

Collection: arbaeen_nawawi_small
Documents: 116


In [4]:
def retrieve(query, top_k=3):
    # تحويل السؤال إلى embedding (بعد التوحيد)
    query_embedding = embedding_model.encode_query(
        normalize_ar(query),
        normalize_embeddings=True
    )

    # البحث عن الأحاديث فقط
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
        where={"type": "hadith"},
        include=["documents", "metadatas", "distances"]
    )

    return results


In [5]:
query = "من هو راوي حديث مراتب الدين؟"

results = retrieve(query, top_k=3)

for i in range(3):
    print("=" * 60)
    print(f"Result {i + 1}")

    metadata = results["metadatas"][0][i]

    print("Hadith Number:", metadata.get("hadith_number"))
    print("Type:", metadata.get("type"))
    print("Title:", metadata.get("title"))
    print("Distance:", results["distances"][0][i])
    print("Content:", results["documents"][0][i][:500])

Result 1
Hadith Number: 2
Type: hadith
Title: مراتب الدين
Distance: 1.3843961954116821
Content: مراتب الدين

عن عمر رضي الله تعالى عنه أقال: بينما نحن جلوس عند رسول الله ذات يوم إذ طلع علينا رجل شديد بياض الثياب شديد سواد الشعر لا يرى عليه أثر السفر ولا يعرفه منا أحد حتى جلس إلى النبي فأسند ركبتيه إلى ركبتيه ووضع كفيه على فخذيه وقال: يا محمد أخبرني عن الإسلام، فقال رسول الله ﷺ: " الإسلام أن تشهد أن لا إله إلا الله وأن محمدا رسول الله، وتقيم الصلاة، وتؤتي الزكاة، وتصوم رمضان، وتحج البيت إن استطعت إليه سبيلا " قال: صدقت. فعجبنا له يسأله ويصدقه، قال: فأخبرني عن الإيمان، قال: " أن تؤمن بالله، 
Result 2
Hadith Number: 11
Type: hadith
Title: اترك ما شككت فيه
Distance: 1.3883506059646606
Content: اترك ما شككت فيه

عن أبي محمد الحسن بن علي بن أبي طالب سبط رسول الله ﷺ وريحانته رضي الله عنهما قال: حفظت من رسول الله ﷺ: (دع ما يريبك إلى ما لا يريبك) رواه الترمذي والنسائي وقال الترمذي: حديث حسن صحيح.
Result 3
Hadith Number: 31
Type: hadith
Title: الزهد في الدنيا
Distance: 1.4376180171966553
Content

In [6]:
query = "أي الحديث اللي بيتكلم عن مراتب الدين؟"

results = retrieve(query, top_k=3)

for i in range(3):
    metadata = results["metadatas"][0][i]

    print("=" * 60)
    print(f"Result {i + 1}")
    print("Hadith Number:", metadata["hadith_number"])
    print("Title:", metadata["title"])
    print("Distance:", results["distances"][0][i])
    print("Content:")
    print(results["documents"][0][i])

Result 1
Hadith Number: 2
Title: مراتب الدين
Distance: 1.0572500228881836
Content:
مراتب الدين

عن عمر رضي الله تعالى عنه أقال: بينما نحن جلوس عند رسول الله ذات يوم إذ طلع علينا رجل شديد بياض الثياب شديد سواد الشعر لا يرى عليه أثر السفر ولا يعرفه منا أحد حتى جلس إلى النبي فأسند ركبتيه إلى ركبتيه ووضع كفيه على فخذيه وقال: يا محمد أخبرني عن الإسلام، فقال رسول الله ﷺ: " الإسلام أن تشهد أن لا إله إلا الله وأن محمدا رسول الله، وتقيم الصلاة، وتؤتي الزكاة، وتصوم رمضان، وتحج البيت إن استطعت إليه سبيلا " قال: صدقت. فعجبنا له يسأله ويصدقه، قال: فأخبرني عن الإيمان، قال: " أن تؤمن بالله، وملائكته، وكتبه، ورسله، واليوم الآخر، وتؤمن بالقدر خيره وشره " قال: صدقت، قال فأخبرني عن الإحسان، قال: " أن تعبد الله كأنك تراه، فإن لم تكن تراه فإنه يراك ". قال: فأخبرني عن الساعة، قال: " ما المسئول عنها بأعلم من السائل " قال: فأخبرني عن أماراتها، قال: " أن تلد الأمة ربتها، وأن ترى الحفاة العراة العالة رعاء الشاء يتطاولون في البنيان " ثم انطلق فلبثت مليا ثم قال: " يا عمر أتدري من السائل؟ " قلت الله ورسوله أعلم قا

In [7]:
# اختبار جودة الاسترجاع: 10 أسئلة (نفس مجموعة الاختبار للمقارنة)
test_retrieval = [
    ("من هو راوي حديث مراتب الدين؟", 2),
    ("أي الحديث اللي بيتكلم عن مراتب الدين؟", 2),
    ("ما هي أركان الإسلام الخمسة؟", 3),
    ("حديث النهي عن الغضب", 16),
    ("ما جزاء معاداة أولياء الله؟", 38),
    ("حديث لا ضرر ولا ضرار", 32),
    ("ما هي علامات الساعة في حديث جبريل؟", 2),
    ("من راوي حديث إنما الأعمال بالنيات؟", 1),
    ("حديث الاستقامة", 21),
    ("ما حكم من أحدث في الدين ما ليس منه؟", 5),
]

correct = 0
for query, expected in test_retrieval:
    results = retrieve(query, top_k=3)
    top1 = results["metadatas"][0][0]["hadith_number"]
    ok = top1 == expected
    correct += ok
    top3 = [(m["hadith_number"], round(float(d), 4))
            for m, d in zip(results["metadatas"][0], results["distances"][0])]
    print(f"Query: {query}")
    print(f"  Expected: {expected} | Top3: {top3} | {'PASS' if ok else 'FAIL'}")

print(f"\nRetrieval accuracy: {correct}/{len(test_retrieval)}")


Query: من هو راوي حديث مراتب الدين؟
  Expected: 2 | Top3: [(2, 1.3844), (11, 1.3884), (31, 1.4376)] | PASS
Query: أي الحديث اللي بيتكلم عن مراتب الدين؟
  Expected: 2 | Top3: [(2, 1.0573), (7, 1.4534), (12, 1.5241)] | PASS
Query: ما هي أركان الإسلام الخمسة؟
  Expected: 3 | Top3: [(3, 0.4423), (2, 0.9731), (7, 1.1317)] | PASS
Query: حديث النهي عن الغضب
  Expected: 16 | Top3: [(16, 0.4844), (9, 0.9255), (30, 0.9999)] | PASS
Query: ما جزاء معاداة أولياء الله؟
  Expected: 38 | Top3: [(38, 0.7614), (41, 1.3361), (24, 1.3814)] | PASS
Query: حديث لا ضرر ولا ضرار
  Expected: 32 | Top3: [(32, 0.5475), (27, 1.1787), (12, 1.2327)] | PASS
Query: ما هي علامات الساعة في حديث جبريل؟
  Expected: 2 | Top3: [(20, 1.4052), (19, 1.4151), (33, 1.4427)] | FAIL
Query: من راوي حديث إنما الأعمال بالنيات؟
  Expected: 1 | Top3: [(1, 0.7476), (37, 1.0476), (31, 1.0952)] | PASS
Query: حديث الاستقامة
  Expected: 21 | Top3: [(21, 0.898), (12, 1.0481), (18, 1.1894)] | PASS
Query: ما حكم من أحدث في الدين ما ليس منه؟
  